# Linear Programming Fundamentals

* Maximize the value of $c^Tx$
Among all vectors $x\in{R}^n$ satifying $Ax \leq b$
* $A$ is $m*n$ real matrix and $c\in R^n$ and $b\in R^m$
* Any vector satisfying all constrains is a **feasible solution**
* Each vector that gives the maximum possible value for the objective function among all feasible vectors is called an **optimal solution**
* A linear program can be **infeasible**,**unbounded**
* convexity and convex polyhedra
* simplex method
* duality theorem
* the ellipsoid method and the interior point method
* Linear Algebra -> Gaussian Elimination -> Affine subspace
* Linear Programming -> simplex method -> system of linear inequalities
* Geometrically, the set of all solutions of a system of linear inequalities is an intersection of finitely many half-spaces in $R^n$. Such a set is called a convex polyhedron.
* Examples : allocate resources, plan production, schedule workers, plan investment portfolios and formulate marketing and military strategies.
* duality theory

# Simplex Method Example with Intuition
$$
\begin{aligned}
\max~ & Z = 3x_1 + 2x_2 \\
\text{s.t. } & 2x_1 + x_2 \le 18 \\
& 2x_1 + 3x_2 \le 42 \\
& 3x_1 + x_2 \le 24 \\
& x_1, x_2 \ge 0
\end{aligned}
$$

**Step 1: Convert to Standard Form (Add Slack Variables)**

Introduce slack variables $s_1, s_2, s_3$ to convert inequalities to equalities:
$$
\begin{cases}
2x_1 + x_2 + s_1 = 18 \\
2x_1 + 3x_2 + s_2 = 42 \\
3x_1 + x_2 + s_3 = 24 \\
x_1, x_2, s_1, s_2, s_3 \ge 0
\end{cases}
$$

**Intuition:** Slack variables represent unused resources. If a constraint is binding, the corresponding slack variable is 0.



**Step 2: Initial Basic Feasible Solution**

Set $x_1 = x_2 = 0$:

$$
s_1 = 18, \quad s_2 = 42, \quad s_3 = 24
$$

This is one corner of the feasible region.

**Step 3: Initial Simplex Tableau**

| Basis | $x_1$ | $x_2$ | $s_1$ | $s_2$ | $s_3$ | RHS |
|-------|----|----|----|----|----|-----|
| $s_1$    | 2  | 1  | 1  | 0  | 0  | 18  |
| $s_2$    | 2  | 3  | 0  | 1  | 0  | 42  |
| $s_3$    | 3  | 1  | 0  | 0  | 1  | 24  |
| $Z$      | -3 | -2 | 0  | 0  | 0  | 0   |

- Negative entries in the $Z$ row indicate which variable can **increase $Z$**.  

**Intuition:** Negative coefficient → increasing that variable improves the objective.

**Step 4: Choose Entering Variable**

Most negative in $Z$ row: $x_1 = -3$ → **enter basis**.

**Step 5: Choose Leaving Variable (Min Ratio Test)**

Compute ratios:

$$
\text{RHS} / \text{Pivot Column Positive Entries}
$$

- $s_1$: $18 / 2 = 9$  
- $s_2$: $42 / 2 = 21$  
- $s_3$: $24 / 3 = 8 \quad \text{(leaving variable)}$

**Intuition:** Choose the constraint that will be hit first as $x_1$ increases to remain feasible.

**Step 6: Pivot**

Pivot on row $s_3$, column $x_1$ (value = 3). Update tableau → $x_1$ replaces $s_3$.

| Basis | $x_1$ | $x_2$ | $s_1$ | $s_2$ | $s_3$ | RHS |
|-------|----|----|----|----|----|-----|
| $s_1$    | 0  | 1/3| 1  | 0  | -2/3| 2  |
| $s_2$    | 0  | 7/3| 0  | 1  | -2/3| 26 |
| $x_1$    | 1  | 1/3| 0  | 0  | 1/3 | 8  |
| $Z$      | 0  | -1 | 0  | 0  | 1   | 24 |


**Step 7: Next Iteration**

- Most negative in $Z$ row: $x_2 = -1$ → **enter basis**.  
- Min ratio test → $s_1$ leaves (2 / (1/3) = 6).  

Pivot → new tableau:

| Basis | $x_1$ | $x_2$ | $s_1$ | $s_2$ | $s_3$ | RHS |
|-------|----|----|----|----|----|-----|
| $x_2$    | 0  | 1  | 3  | 0  | -2  | 6  |
| $s_2$    | 0  | 0  | -7 | 1  | 4   | 4  |
| $x_1$    | 1  | 0  | -1 | 0  | 1   | 6  |
| $Z$      | 0  | 0  | 3  | 0  | -1  | 30 |

No negative entries in the $Z$ row → **optimal solution reached**

**Step 8: Final Solution**

$$
x_1 = 6, \quad x_2 = 6, \quad Z_{\max} = 30
$$

- Slack variables show remaining resources:

$$
s_1 = 0, \quad s_2 = 4, \quad s_3 = 0
$$

**Intuition:** Simplex walked along the **edges of the feasible polytope**, moving from one corner to the next, improving $Z$ at each step until optimal.


# Simplex Implementation

In [5]:
import numpy as np

def simplex(c, A, b):
    """
    Simplex algorithm for standard form LP:
        max c^T x
        s.t. Ax = b, x >= 0
    """
    m, n = A.shape
    tableau = np.zeros((m+1, n+m+1))
    
    # Build initial tableau with slack variables
    tableau[:m, :n] = A
    tableau[:m, n:n+m] = np.eye(m)
    tableau[:m, -1] = b
    tableau[-1, :n] = -c  # Objective row
    
    basis = list(range(n, n+m))  # Initial basic variables (slacks)
    
    while True:
        # Step 1: Choose entering variable (most negative in objective row)
        col = np.argmin(tableau[-1, :-1])
        if tableau[-1, col] >= 0:
            break  # Optimal solution reached
        
        # Step 2: Choose leaving variable (min ratio test)
        ratios = [tableau[i,-1]/tableau[i,col] if tableau[i,col]>0 else np.inf for i in range(m)]
        row = np.argmin(ratios)
        
        # Step 3: Pivot
        pivot = tableau[row, col]
        tableau[row] /= pivot
        for i in range(m+1):
            if i != row:
                tableau[i] -= tableau[i, col] * tableau[row]
        
        basis[row] = col  # Update basic variable
    
    # Extract solution
    x = np.zeros(n+m)
    for i in range(m):
        x[basis[i]] = tableau[i,-1]
    
    return x[:n], tableau[-1,-1]

In [6]:
# Example LP
c = np.array([3, 2])
A = np.array([[2,1],[2,3],[3,1]])
b = np.array([18,42,24])

solution, obj = simplex(c, A, b)

# Print final optimal solution
print("Optimal x:", solution)
print("Optimal Z:", obj)

Optimal x: [ 3. 12.]
Optimal Z: 33.0
